## Reduce zip codes geojson file to only relevant zips

In [1]:
import json
from pathlib import Path

import geopandas as gpd

zip_path = Path("georgia-zip-codes.geojson")
municipality_path = Path("Municipality.geojson")
buffer_miles = 1

zip_codes = gpd.read_file(zip_path)
municipalities = gpd.read_file(municipality_path)

# Municipality.geojson contains Savannah, Tybee Island, the other Chatham
# municipalities, and the county's unincorporated area. Their union is the
# full county footprint. Use a projected CRS so the one-mile buffer is exact.
working_crs = "EPSG:26917"  # NAD83 / UTM zone 17N, meters
county_footprint = municipalities.to_crs(working_crs).geometry.union_all()
county_plus_buffer = county_footprint.buffer(buffer_miles * 1609.344)

zip_codes_projected = zip_codes.to_crs(working_crs)
near_chatham = zip_codes.loc[zip_codes_projected.intersects(county_plus_buffer)].copy()
near_chatham = near_chatham.sort_values("code")

# Record how deeply each ZIP reaches into each jurisdiction. The maps use
# these measurements for an adjustable boundary-sliver tolerance without
# having to run expensive polygon intersections in the browser.
municipalities_projected = municipalities.to_crs(working_crs)
jurisdiction_depths = []
for zip_geometry in zip_codes_projected.loc[near_chatham.index].geometry:
    depths = {}
    for _, jurisdiction in municipalities_projected.iterrows():
        if not zip_geometry.intersects(jurisdiction.geometry):
            continue
        low, high = 0.0, 50000.0
        for _ in range(18):
            midpoint = (low + high) / 2
            interior = jurisdiction.geometry.buffer(-midpoint)
            if not interior.is_empty and zip_geometry.intersects(interior):
                low = midpoint
            else:
                high = midpoint
        depths[str(jurisdiction["NAME"])] = round(low / 0.9144, 1)
    jurisdiction_depths.append(json.dumps(depths, separators=(",", ":")))
near_chatham["jurisdiction_depth_yards"] = jurisdiction_depths

# Replace the statewide file with only ZIP polygons that touch Chatham County
# or its one-mile buffer. Keep WGS84 coordinates for use in the web maps.
near_chatham.to_crs("EPSG:4326").to_file(zip_path, driver="GeoJSON")
print(f"Kept {len(near_chatham)} ZIP codes: {', '.join(near_chatham['code'].astype(str))}")


C:\Users\micha\anaconda3\Lib\site-packages\pyogrio\core.py:35: RuntimeWarning: Could not detect GDAL data files.  Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


Kept 20 ZIP codes: 31302, 31308, 31312, 31318, 31322, 31324, 31326, 31328, 31401, 31404, 31405, 31406, 31407, 31408, 31409, 31410, 31411, 31415, 31419, 31421
